# Extract `boot.img` from a SamFW firmware zip

The firmware zip is ~7.5 GB and almost all of it is `super.img`. Only
`boot.img` (~80 MB) is needed, because it holds the ramdisk the kernel
requires in order to have an `/init` at all.

This reads the zip's central directory, walks straight to the AP tar, and
streams through it until `boot.img` appears. Nothing else is ever written to
disk, so peak usage is about the size of `boot.img` rather than 7.5 GB.

**Run the cells in order.** At the end it saves `boot.img` to your Drive.

> Set the runtime to **None / CPU** — no GPU is needed and it just costs quota.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive/MyDrive')

In [ ]:
import os

# Path inside your Drive to the firmware zip. Edit if yours differs.
ZIP_PATH = '/content/drive/MyDrive/GrapheneOSA245F(A24)/SAMFW.COM_SM-A245F_TUR_A245FXXS9DYG1_fac.zip'

if not os.path.isfile(ZIP_PATH):
    # list what is actually there so the path can be corrected
    d = os.path.dirname(ZIP_PATH)
    print('not found:', ZIP_PATH)
    print('contents of', d, ':')
    if os.path.isdir(d):
        for f in os.listdir(d):
            p = os.path.join(d, f)
            print('   %10.2f MB  %s' % (os.path.getsize(p)/1048576, f))
    else:
        print('   directory does not exist either')
else:
    print('found: %.2f GiB' % (os.path.getsize(ZIP_PATH)/1024**3))

In [ ]:
import hashlib, os, tarfile, zipfile

OUT = '/content/boot.img'

def find_ap(zf):
    names = zf.namelist()
    ap = [n for n in names
          if os.path.basename(n).upper().startswith('AP')
          and (n.lower().endswith('.tar.md5') or n.lower().endswith('.tar'))]
    if ap:
        return ap[0]
    for n in names:
        low = n.lower()
        if not (low.endswith('.tar') or low.endswith('.tar.md5')):
            continue
        base = os.path.basename(n).upper()
        if base.startswith(('BL', 'CP', 'CSC', 'HOME_CSC', 'USERDATA')):
            continue
        return n
    raise SystemExit('no AP tar in zip: ' + repr(names[:40]))

with zipfile.ZipFile(ZIP_PATH) as zf:
    print('zip members:', len(zf.namelist()))
    ap = find_ap(zf)
    print('AP tar: %s (%.2f GiB)' % (ap, zf.getinfo(ap).file_size / 1024**3))

    # r| streams and stops at the end-of-archive marker, so the .md5 tail
    # appended by samfw is ignored without needing to seek.
    with zf.open(ap) as raw, tarfile.open(fileobj=raw, mode='r|') as tar:
        got = False
        for entry in tar:
            if not entry.isfile():
                continue
            if os.path.basename(entry.name) != 'boot.img':
                continue
            digest = hashlib.sha256()
            n = 0
            with tar.extractfile(entry) as src, open(OUT, 'wb') as dst:
                while True:
                    chunk = src.read(1 << 20)
                    if not chunk:
                        break
                    dst.write(chunk)
                    digest.update(chunk)
                    n += len(chunk)
            print('boot.img: %d bytes (%.2f MiB)' % (n, n / 1048576))
            print('sha256  :', digest.hexdigest())
            got = True
            break
        if not got:
            raise SystemExit('no boot.img inside the AP tar')

print('partition size is 83,886,080 bytes for a24')
print('match:', os.path.getsize(OUT) == 83886080)

In [ ]:
# Sanity check the ramdisk before saving: this is the whole point of the job.
import struct

raw = open('/content/boot.img', 'rb').read()
assert raw[:8] == b'ANDROID!', 'not an Android boot image'

kernel_size, = struct.unpack_from('<I', raw, 8)
ramdisk_size, = struct.unpack_from('<I', raw, 16)
page, = struct.unpack_from('<I', raw, 36)
print('header version:', struct.unpack_from('<I', raw, 40)[0])
print('page size     :', page)
print('kernel size   : %.2f MiB' % (kernel_size / 1048576))
print('ramdisk size  : %.2f MiB' % (ramdisk_size / 1048576))

def pages(n):
    return (n + page - 1) // page

off = page + pages(kernel_size) * page
rd = raw[off:off + ramdisk_size]
print('ramdisk magic :', rd[:4].hex())

if ramdisk_size < 1048576:
    print('WARNING: ramdisk under 1 MiB - this is not the stock initramfs')
else:
    print('OK: this is a real ramdisk, %d bytes' % ramdisk_size)

In [ ]:
# Save next to the firmware zip so it is easy to find.
import shutil

dest = '/content/drive/MyDrive/GrapheneOSA245F(A24)/boot_A245FXXS9DYG1_stock.img'
shutil.copyfile('/content/boot.img', dest)
print('saved to:', dest)
print('size    : %.2f MiB' % (os.path.getsize(dest) / 1048576))
print()
print('Now download that file from Drive and hand it over. It is ~80 MB,',
      'which is small enough to attach directly.')